In [ ]:
import sys

import pandas as pd
import plotly
import plotly.graph_objects as go
import kaleido

from plotly.subplots import make_subplots

In [ ]:
#REQUIRED FUNCTIONS

In [ ]:
def plot_all_sankey_single(mpsat_results, polymer_types, height_par):
    num_polymers = len(polymer_types)

    pt2panel_name = {'PE' : '(a)', 'PP' : '(b)', 'PS' : '(c)'}
    
    # Define A4 width in pixels (≈ 800px for good fit)
    fig_width = 350  # Adjusted for A4 width

    # Adjust padding & thickness for better spacing
    pad_settings = {"PE": 20, "PP": 15, "PS": 10}
    thickness_settings = {"PE": 10, "PP": 10, "PS": 10}

    # Set different heights for each polymer type
    height_settings = {"PE": 400, "PP": 200, "PS": 100}  # Adjust these values based on your preference

    # Create a 1-row, multiple-column subplot layout
    fig = make_subplots(
        rows=1, cols=num_polymers,  # All plots in one row 
        specs=[[{"type": "sankey"}] * num_polymers], 
        horizontal_spacing=0.1,  # Reduce spacing for compact layout
        subplot_titles=[f"{pt2panel_name[polymer_types[0]]}"]
    )

    for i, polymer_type in enumerate(polymer_types):
        # Filter data for the polymer type
        mpsat_results_filtered = mpsat_results[mpsat_results["true"] == polymer_type]
        class_counts = mpsat_results_filtered["predicted"].value_counts()

        if class_counts.empty:
            print(f"No data for {polymer_type}, skipping...")
            continue

        # Define source & target
        source = [0] * len(class_counts)
        target = list(range(1, len(class_counts) + 1))
        values = class_counts.values

        # Labels for nodes
        labels = [f"{polymer_type} ({mpsat_results_filtered.shape[0]})"] + \
                 [f"{cls} ({count})" for cls, count in zip(class_counts.index, values)]

        # Adjust node positioning (x: left-right placement)
        node_positions = [0.3] + [0.7] * len(class_counts)

        # Create Sankey trace
        sankey_trace = go.Sankey(
            node=dict(
                pad=pad_settings.get(polymer_type, 10),
                thickness=thickness_settings.get(polymer_type, 8),
                line=dict(color="black", width=0.1),
                label=labels,
                x=node_positions
            ),
            link=dict(
                source=source,
                target=target,
                value=values
            )
        )

        # Add trace to subplot
        fig.add_trace(sankey_trace, row=1, col=i + 1)

    # Update layout for each subplot with appropriate height for each polymer
    fig.update_layout(
        font_size=10,
        width=fig_width,
        height=height_par,  # Use the max height to maintain consistency
        autosize=False,
        title_x=0,  # Move titles to the left (horizontal position)
        title_y=1,  # Move titles closer to the plot (vertical position)
        title_font=dict(size=12),  # Optional: Adjust font size of the titles
        showlegend=False  # Optional: Hide the legend if not needed
    )

    # Save as PNG
    if len(polymer_types) == 1:
        file_name = f"{polymer_types[0]}_sankey_plots_compact.png"
    else:
        print('DEFINE FILENAME')
        sys.exit()
    fig.write_image(file_name, scale=3)
    print(f"Saved: {file_name}")

    # Show figure
    #fig.show()

In [ ]:
#END REQUIRED FUNCTIONS

In [ ]:
#MAIN

In [ ]:
red_db_file_baseline  = "../../4_baseline_correction/MICROSCAN_database_baseline_corrected.csv"
red_db_baseline = pd.read_csv(red_db_file_baseline)
red_db_baseline.head()

In [ ]:
red_db_file  = "../../2_compiling_unified_database/MICROSCAN_database.csv"
red_db = pd.read_csv(red_db_file)
red_db.head()

In [ ]:
# Load CNN1D predictions
df = pd.read_csv('../../3_applying_CNN1D/manual_and_CNN1D_classification.csv')
df.head()

In [ ]:
#DRAW SANKEY

In [ ]:
polymer_list = ['PE', 'PP', 'PS']
heights      = [500, 451, 316]
for i,j in zip(polymer_list,heights):
    plot_all_sankey_single(df, [i], j)